In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")  # notebook lives one level under project root

files = list(RAW_DIR.glob("*.csv"))
print(f"Found {len(files)} coin files")

dfs = []
for f in files:
    coin_id = f.stem  # filename without ".csv" — e.g. "bitcoin"
    df = pd.read_csv(f, parse_dates=["date"])
    df["coin"] = coin_id
    dfs.append(df)

all_data = pd.concat(dfs, ignore_index=True)
all_data.head()

Found 205 coin files


,date,price,volume,coin
0,2026-08-20,0.021402,5418068.0,aligned
1,2026-08-21,0.018251,26302338.0,aligned
2,2026-08-22,0.016803,17537841.0,aligned
3,2026-08-23,0.015204,17226034.0,aligned
4,2026-08-24,0.014786,18894804.0,aligned


In [2]:
price_panel = all_data.pivot(index="date", columns="coin", values="price")
price_panel.shape

(731, 205)

In [3]:
volume_panel = all_data.pivot(index="date", columns="coin", values="volume")

In [4]:
coverage = price_panel.notna().mean()
coverage.sort_values().head(10)  # worst-covered coins

coin
little-john       0.001368
royal-euro        0.001368
hyperliquid       0.001368
aster-2           0.001368
teller            0.001368
interfold         0.001368
official-trump    0.001368
rain              0.001368
dgrid-ai          0.004104
united-stables    0.006840
dtype: float64

In [5]:
COVERAGE_THRESHOLD = 0.90
good_coins = coverage[coverage >= COVERAGE_THRESHOLD].index
print(f"{len(good_coins)} of {len(coverage)} coins meet the {COVERAGE_THRESHOLD:.0%} bar")

price_panel = price_panel[good_coins]
volume_panel = volume_panel[good_coins]

134 of 205 coins meet the 90% bar


In [6]:
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(exist_ok=True)
price_panel.to_csv(OUT_DIR / "prices.csv")
volume_panel.to_csv(OUT_DIR / "volumes.csv")